# Average RMSF of the whole CESA1 complex

Computes per-residue backbone RMSF for each of the 8 replicas after aligning
to the common reference structure (backbone.pdb), then aggregates into a
single average RMSF profile.

**Inputs:**
- Reference: `backbone.pdb`
- Per-replica backbone trajectories: `traj1.xtc` → `traj8.xtc`

**Outputs:**
- `rmsf_per_residue_CESA1.csv` — per-residue RMSF per replica
- `rmsf_per_residue_CESA1_summary.csv` — mean / std / min / max per replica + overall
- `rmsf_per_residue_CESA1.png` — RMSF vs residue plot
- `rmsf_top10_flexible_CESA1.csv` — top 10 most flexible residues
- `rmsf_top10_rigid_CESA1.csv` — top 10 most rigid residues (binding candidates)

In [ ]:
# ============================================================
# CELL 1 — Install MDAnalysis if needed
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'MDAnalysis', '-q'])
print('MDAnalysis ready')

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

print('MDAnalysis', mda.__version__)

In [ ]:
# ============================================================
# CELL 3 — Configuration
# Same paths as RMSD notebook — no changes needed
# ============================================================

BASE_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/result_CESA1_ClusPro/CESA1_prod/Backbone_xtc')

REPLICAS = ['traj1', 'traj2', 'traj3', 'traj4', 'traj5', 'traj6', 'traj7', 'traj8']
REF_PDB  = BASE_DIR / 'backbone.pdb'    # same fixed reference as RMSD
TRAJ_FMT = '{R}.xtc'
ATOM_SEL = 'backbone'                   # backbone atoms for RMSF

# Output files
OUT_DIR           = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/result_CESA1_ClusPro/run')
OUT_CSV_RESIDUES  = OUT_DIR / 'rmsf_per_residue_CESA1.csv'
OUT_CSV_STATS     = OUT_DIR / 'rmsf_per_residue_CESA1_summary.csv'
OUT_PNG           = OUT_DIR / 'rmsf_per_residue_CESA1.png'
OUT_TOP_FLEXIBLE  = OUT_DIR / 'rmsf_top10_flexible_CESA1.csv'
OUT_TOP_RIGID     = OUT_DIR / 'rmsf_top10_rigid_CESA1.csv'

# Verify all files exist
assert REF_PDB.exists(), f'Reference PDB not found: {REF_PDB}'
for R in REPLICAS:
    p = BASE_DIR / TRAJ_FMT.format(R=R)
    assert p.exists(), f'Missing trajectory: {p}'
    print(f'  {R}: {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')
print(f'Reference: {REF_PDB.name}')

In [ ]:
# ============================================================
# CELL 4 — Per-replica RMSF (complete clean version)
# ============================================================

results = {}

for R in REPLICAS:
    traj_path = BASE_DIR / TRAJ_FMT.format(R=R)
    print(f'-- {R}: loading {traj_path.name}')

    u   = mda.Universe(str(REF_PDB), str(traj_path))
    ref = mda.Universe(str(REF_PDB))

    # Align trajectory to reference
    aligner = align.AlignTraj(u, ref, select=ATOM_SEL, in_memory=True).run()

    # Calculate RMSF per atom
    sel       = u.select_atoms(ATOM_SEL)
    rmsf_calc = rms.RMSF(sel).run()

    # Average RMSF per residue using chain+resid
    res_ids    = []
    res_names  = []
    res_rmsf   = []
    res_labels = []

    for i, res in enumerate(sel.residues):
        atom_indices = np.where(
            (sel.resids == res.resid) &
            (sel.segids == res.segid)
        )[0]
        if len(atom_indices) > 0:
            mean_rmsf = rmsf_calc.results.rmsf[atom_indices].mean()
            res_ids.append(i + 1)
            res_names.append(res.resname)
            res_rmsf.append(mean_rmsf)
            res_labels.append(f'{res.segid}_{res.resid}')

    res_ids    = np.array(res_ids)
    res_names  = np.array(res_names)
    res_rmsf   = np.array(res_rmsf)
    res_labels = np.array(res_labels)

    results[R] = (res_ids, res_names, res_rmsf, res_labels)

    print(f'   residues={len(res_ids)}  '
          f'mean={res_rmsf.mean():.3f} Å  '
          f'max={res_rmsf.max():.3f} Å  '
          f'sorted={np.all(np.diff(res_ids) > 0)}')

In [ ]:
u = mda.Universe(str(REF_PDB))
sel = u.select_atoms('backbone')
print('Unique segment IDs:', np.unique(sel.segids))
print('Unique chain IDs:', np.unique(sel.chainIDs) if hasattr(sel, 'chainIDs') else 'N/A')
print('Resid range:', sel.resids.min(), '→', sel.resids.max())
print('Total residues:', len(sel.residues))

In [ ]:
ref_ids, ref_resnames, _, ref_labels = results['traj1']

rmsf_stack = []
for R in REPLICAS:
    ids, _, rmsf_vals, _ = results[R]
    rmsf_stack.append(rmsf_vals)

rmsf_stack = np.vstack(rmsf_stack)
rmsf_mean  = np.nanmean(rmsf_stack, axis=0)
rmsf_std   = np.nanstd(rmsf_stack, axis=0)

print(f'Number of residues : {len(ref_ids)}')
print(f'Mean RMSF (all)    : {rmsf_mean.mean():.3f} ± {rmsf_mean.std():.3f} Å')
print(f'Max RMSF           : {rmsf_mean.max():.3f} Å at index {ref_ids[rmsf_mean.argmax()]}')
print(f'Min RMSF           : {rmsf_mean.min():.3f} Å at index {ref_ids[rmsf_mean.argmin()]}')
print(f'Label at max RMSF  : {ref_labels[rmsf_mean.argmax()]}')
print(f'Label at min RMSF  : {ref_labels[rmsf_mean.argmin()]}')

In [ ]:
u = mda.Universe(str(REF_PDB))
sel = u.select_atoms('backbone')
cumulative = 0
boundaries = {}
for seg in ['A', 'B', 'C', 'D']:
    chain_res = sel.select_atoms(f'segid {seg}').residues
    boundaries[seg] = (cumulative + 1, cumulative + len(chain_res))
    cumulative += len(chain_res)
    print(f'Chain {seg}: {len(chain_res)} residues  '
          f'→  sequential index {boundaries[seg][0]} to {boundaries[seg][1]}')
print(f'\nTotal: {cumulative} residues')
print(f'boundaries dict: {boundaries}')

In [ ]:
# ============================================================
# CELL 7 — Summary table per replica
# ============================================================

summary_rows = []
for R, (_, _, rmsf_vals, _) in results.items():
    summary_rows.append({
        'replica' : R,
        'n_res'   : len(rmsf_vals),
        'mean_A'  : float(np.mean(rmsf_vals)),
        'std_A'   : float(np.std(rmsf_vals)),
        'min_A'   : float(np.min(rmsf_vals)),
        'max_A'   : float(np.max(rmsf_vals)),
    })

# Overall average
summary_rows.append({
    'replica' : 'AVERAGE (across replicas)',
    'n_res'   : len(ref_ids),
    'mean_A'  : float(rmsf_mean.mean()),
    'std_A'   : float(rmsf_mean.std()),
    'min_A'   : float(rmsf_mean.min()),
    'max_A'   : float(rmsf_mean.max()),
})

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_CSV_STATS, index=False)
print(f'Wrote {OUT_CSV_STATS.name}')
df_summary.round({'mean_A': 3, 'std_A': 3, 'min_A': 3, 'max_A': 3})

In [ ]:
# ============================================================
# CELL 8 — Top 10 most flexible and most rigid residues
#
# Most flexible = high RMSF = loops, termini
# Most rigid    = low RMSF  = binding interface candidates
# ============================================================

# Get resnames from traj1
ref_resnames = results['traj1'][1]

df_avg = pd.DataFrame({
    'resid'   : ref_ids,
    'resname' : ref_resnames,
    'rmsf_mean_A' : rmsf_mean,
    'rmsf_std_A'  : rmsf_std
})

# Top 10 most flexible
top_flexible = df_avg.nlargest(10, 'rmsf_mean_A')
top_flexible.to_csv(OUT_TOP_FLEXIBLE, index=False)
print('Top 10 MOST FLEXIBLE residues (high RMSF):')
print(top_flexible.to_string(index=False))

print()

# Top 10 most rigid (binding candidates)
top_rigid = df_avg.nsmallest(10, 'rmsf_mean_A')
top_rigid.to_csv(OUT_TOP_RIGID, index=False)
print('Top 10 MOST RIGID residues (low RMSF — binding candidates):')
print(top_rigid.to_string(index=False))

In [ ]:
# ============================================================
# CELL 9 — Plot RMSF profile (Q1 Publication Grade)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Professional color palette (muted, publication-friendly colors)
colors = {
    'traj1': '#1f77b4', 'traj2': '#2ca02c', 'traj3': '#d62728',
    'traj4': '#9467bd', 'traj5': '#8c564b', 'traj6': '#e377c2',
    'traj7': '#7f7f7f', 'traj8': '#bcbd22'
}

# Set up the figure with standard publication-ready font and styling
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)

# 1. Plot individual replicas with thin, semi-transparent lines
for R in REPLICAS:
    ids, _, rmsf_vals, _ = results[R]
    ax.plot(ids, rmsf_vals, color=colors.get(R, '#7f7f7f'),
            lw=0.6, alpha=0.35, label=R)

# 2. Plot the mean profile with a thick, solid black line
ax.plot(ref_ids, rmsf_mean, color='black', lw=1.8,
        label='Mean across replicas', zorder=5)

# 3. Plot the Standard Deviation (SD) shade
ax.fill_between(
    ref_ids,
    rmsf_mean - rmsf_std,
    rmsf_mean + rmsf_std,
    color='black', alpha=0.12, label='±1 SD', zorder=4
)

# 4. Chain Boundaries & Labels (highly recommended for Q1 journals)
chain_boundaries = {
    'Chain A': (1, 619),
    'Chain B': (620, 1420),
    'Chain C': (1421, 2221),
    'Chain D': (2222, 3022)
}

# Draw vertical boundary lines
boundary_positions = [619.5, 1420.5, 2221.5]
for pos in boundary_positions:
    ax.axvline(x=pos, color='#d3d3d3', linestyle='--', lw=1.2, zorder=1)

# Add Chain Text Labels in the upper region of each panel
for chain_name, (start, end) in chain_boundaries.items():
    center = (start + end) / 2
    # Place label at 85% of maximum y-value to avoid overlapping data
    y_pos = ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 25.0
    ax.text(center, y_pos, chain_name, fontsize=12, weight='bold',
            ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Axes & Layout refinement
ax.set_xlabel('Sequential Residue Index', fontsize=12, labelpad=8)
ax.set_ylabel('Backbone RMSF (Å)', fontsize=12, labelpad=8)
ax.set_title('CESA1 / HVGlu3 Complex - Backbone RMSF Profile', fontsize=13, weight='bold', pad=15)

# X-ticks based on chain centers & boundaries for easier navigation
ax.set_xlim(0, 3023)
ax.set_ylim(0, max(rmsf_mean + rmsf_std) * 1.1)

# Clean legend
ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='#e0e0e0', framealpha=0.9, ncol=4, fontsize=9.5)
ax.grid(True, which='both', color='#f0f0f0', linestyle='-', linewidth=0.5, zorder=0)

# Remove top and right spines for a clean, modern look (standard in Q1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()

# Save in high-res PNG and vector PDF format for publication
OUT_PDF = OUT_PNG.with_suffix('.pdf')
fig.savefig(OUT_PNG, dpi=300)
fig.savefig(OUT_PDF, dpi=300)
print(f'Wrote {OUT_PNG.name} (PNG, 300 DPI)')
print(f'Wrote {OUT_PDF.name} (Vector PDF)')
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Plot RMSF profile (Q1 Publication Grade)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

colors = {
    'traj1': '#1f77b4', 'traj2': '#2ca02c', 'traj3': '#d62728',
    'traj4': '#9467bd', 'traj5': '#8c564b', 'traj6': '#e377c2',
    'traj7': '#7f7f7f', 'traj8': '#bcbd22'
}

plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

# Extra top margin to fit chain labels OUTSIDE the plot
fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)

# 1. Individual replicas
for R in REPLICAS:
    ids, _, rmsf_vals, _ = results[R]
    ax.plot(ids, rmsf_vals, color=colors.get(R, '#7f7f7f'),
            lw=0.6, alpha=0.35, label=R)

# 2. Mean profile
ax.plot(ref_ids, rmsf_mean, color='black', lw=1.8,
        label='Mean across replicas', zorder=5)

# 3. SD shading
ax.fill_between(
    ref_ids,
    rmsf_mean - rmsf_std,
    rmsf_mean + rmsf_std,
    color='black', alpha=0.12, label='±1 SD', zorder=4
)

# 4. Chain boundaries and borders
chain_boundaries = {
    'Chain A\n(HVGlu3)' : (1,    619),
    'Chain B\n(CESA1)'  : (620,  1420),
    'Chain C\n(CESA1)'  : (1421, 2221),
    'Chain D\n(CESA1)'  : (2222, 3022)
}

# Y axis limits — set before drawing rectangles
y_max = float(np.max(rmsf_mean + rmsf_std)) * 1.15
ax.set_ylim(0, y_max)
ax.set_xlim(0, 3023)

# Background colors for each chain region
chain_colors = {
    'Chain A\n(HVGlu3)' : '#fff7e6',   # light orange for HVGlu3
    'Chain B\n(CESA1)'  : '#e8f4f8',   # light blue
    'Chain C\n(CESA1)'  : '#e8f4f8',   # light blue
    'Chain D\n(CESA1)'  : '#e8f4f8',   # light blue
}

for chain_name, (start, end) in chain_boundaries.items():
    # Draw filled rectangle with border inside the plot
    rect = mpatches.FancyBboxPatch(
        (start, 0),
        end - start,
        y_max,
        boxstyle='square,pad=0',
        linewidth=1.2,
        edgecolor='#888888',
        facecolor=chain_colors[chain_name],
        alpha=0.25,
        zorder=0
    )
    ax.add_patch(rect)

    # Draw vertical boundary lines
    ax.axvline(x=start, color='#888888', linestyle='-', lw=1.0, zorder=1)
    ax.axvline(x=end,   color='#888888', linestyle='-', lw=1.0, zorder=1)

    # Chain labels OUTSIDE plot — above the top spine
    center = (start + end) / 2
    ax.annotate(
        chain_name,
        xy=(center, y_max),
        xycoords='data',
        xytext=(center, y_max * 1.08),
        textcoords='data',
        fontsize=10,
        weight='bold',
        ha='center',
        va='bottom',
        annotation_clip=False,   # allows drawing outside axes
        bbox=dict(
            facecolor='white',
            edgecolor='#888888',
            boxstyle='round,pad=0.3',
            linewidth=0.8
        )
    )

# Axes
ax.set_xlabel('Sequential Residue Index', fontsize=12, labelpad=8)
ax.set_ylabel('Backbone RMSF (Å)', fontsize=12, labelpad=8)

# Title OUTSIDE — above chain labels using suptitle
fig.suptitle(
    'CESA1 / HvGlu3 Complex',
    fontsize=13,
    weight='bold',
    y=1.08   # pushes title above chain labels
)

# Legend
ax.legend(
    loc='upper right',
    frameon=True,
    facecolor='white',
    edgecolor='#e0e0e0',
    framealpha=0.9,
    ncol=4,
    fontsize=9.5
)

ax.grid(True, which='both', color='#f0f0f0',
        linestyle='-', linewidth=0.5, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.subplots_adjust(top=0.78)   # make room for chain labels and title above

# Save
OUT_PDF = OUT_PNG.with_suffix('.pdf')
fig.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
fig.savefig(OUT_PDF, dpi=300, bbox_inches='tight')
print(f'Wrote {OUT_PNG.name} (PNG, 300 DPI)')
print(f'Wrote {OUT_PDF.name} (Vector PDF)')
plt.show()

In [ ]:
# Print max RMSF per replica to find the outlier
for R in REPLICAS:
    _, _, rmsf_vals, _ = results[R]
    print(f'{R}: mean={rmsf_vals.mean():.2f} Å  max={rmsf_vals.max():.2f} Å  '
          f'at resid={results[R][0][rmsf_vals.argmax()]}')

In [ ]:
# ============================================================
# CELL 10 — Headline numbers
# ============================================================

overall_mean = rmsf_mean.mean()
overall_std  = rmsf_mean.std()
most_flex_resid = ref_ids[rmsf_mean.argmax()]
most_rigid_resid = ref_ids[rmsf_mean.argmin()]

print(f'Average RMSF of the whole complex (mean across {len(REPLICAS)} replicas):')
print(f'    {overall_mean:.3f} ± {overall_std:.3f} Å')
print()
print(f'Most flexible residue : {most_flex_resid} ({ref_resnames[rmsf_mean.argmax()]})  '
      f'RMSF = {rmsf_mean.max():.3f} Å')
print(f'Most rigid residue    : {most_rigid_resid} ({ref_resnames[rmsf_mean.argmin()]})  '
      f'RMSF = {rmsf_mean.min():.3f} Å')